In [30]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv('recipes.csv')
texts = df['text']
y_tr = (df['label'] == 'savoury').astype(int).values  # 0 — десерты, 1 — мясо/рыба/закуски

tv =TfidfVectorizer(stop_words='english') 
X = tv.fit_transform(texts).toarray() #берёт те же счётчики, но умножает их на вес редкости слова.

In [7]:
import numpy as np
from sklearn.metrics import pairwise_distances
from sklearn.cluster import AffinityPropagation

S = -pairwise_distances(X, metric='sqeuclidean')
med = np.median(S)
print(med)
for k in [0.5, 1, 1.5, 1.7, 2]:
    ap = AffinityPropagation(preference=med * k, random_state=0).fit(X)
    print(k, len(ap.cluster_centers_indices_))

-1.8329086862108046
0.5 20
1 4
1.5 2
1.7 1
2 1


In [ ]:
from sklearn.metrics import rand_score, adjusted_rand_score
from sklearn.metrics import silhouette_score
from sklearn.cluster import Birch

ap = AffinityPropagation(damping = 0.5, random_state=0, max_iter = 200, preference = med * 1.5)
ap.fit(X) 
lab = ap.labels_
br = Birch(n_clusters=2, threshold=0.8, branching_factor= 50) #threshold -- максимальный радиус подкластера, то есть среднее расстояние от его точек до центра; branching_factor -- максимальное число детей у одного узла: подкластеров в листе или дочерних узлов во внутреннем узле.
#На каждом шаге находятся две самые близкие группы и склеиваются в одну. Это повторяется, пока групп не останется n_clusters. Cклеиваются те две группы, после объединения которых разброс внутри групп вырастет меньше всего
br.fit(X)
lab_b = br.labels_

ri = rand_score(y_tr, lab)
ari = adjusted_rand_score(y_tr, lab)
sil =  silhouette_score(X, lab)

rib = rand_score(y_tr, lab_b)
arib = adjusted_rand_score(y_tr, lab_b)
silb =  silhouette_score(X, lab_b)

print('AffinityPropagation')
print('кластеров:', len(ap.cluster_centers_indices_))
print('RI :', ri)
print('ARI:', ari)
print('Sil:',sil)

print('Birch')
print('подкластеров:', len(br.subcluster_centers_))
print('кластеров:', len(set(lab)))
print('RI :', rib)
print('ARI:', arib)
print('Sil:',silb)

AffinityPropagation
кластеров: 2
RI : 0.9
ARI: 0.7995558023320377
Sil: 0.02955096176397351
Birch
подкластеров: 16
кластеров: 2
RI : 1.0
ARI: 1.0
Sil: 0.036446838528767414


In [21]:
from sklearn.decomposition import PCA
import plotly.graph_objects as go

X3 = PCA(n_components=3).fit_transform(X)
centers = ap.cluster_centers_indices_     # номера рецептов-центров
C3 = X3[centers]                          # их координаты на графике

# палочки: от каждой точки к своему центру
lx, ly, lz = [], [], []
for p, c in zip(X3, C3[lab]):
    lx += [p[0], c[0], None]
    ly += [p[1], c[1], None]
    lz += [p[2], c[2], None]

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=lx, y=ly, z=lz, mode='lines',
                           line=dict(color='black', width=2), hoverinfo='skip'))
fig.add_trace(go.Scatter3d(x=X3[:, 0], y=X3[:, 1], z=X3[:, 2], mode='markers',
                           marker=dict(size=5, color='limegreen', line=dict(color='darkgreen', width=1)),
                           text=df['text'].str[:40]))
fig.add_trace(go.Scatter3d(x=C3[:, 0], y=C3[:, 1], z=C3[:, 2], mode='markers',
                           marker=dict(size=10, color='red', line=dict(color='darkred', width=1)),
                           text=df['text'][centers].str[:40]))
fig.update_layout(height=700, width=900, margin=dict(l=0, r=0, t=40, b=0), showlegend=False,
                  title=f'RI = {ri:.3f}   ARI = {ari:.3f}    Silhouette  = {sil:.3f}')
fig.show()

In [25]:
C3_b = np.array([X3[lab_b == k].mean(axis=0) for k in range(lab_b.max() + 1)]) #центроиды классов LS/N

lx_b, ly_b, lz_b = [], [], []
for p, c in zip(X3, C3_b[lab_b]):
    lx_b += [p[0], c[0], None]
    ly_b += [p[1], c[1], None]
    lz_b += [p[2], c[2], None]

fig_b = go.Figure()
fig_b.add_trace(go.Scatter3d(x=lx_b, y=ly_b, z=lz_b, mode='lines',
                             line=dict(color='black', width=2), hoverinfo='skip'))
fig_b.add_trace(go.Scatter3d(x=X3[:, 0], y=X3[:, 1], z=X3[:, 2], mode='markers',
                             marker=dict(size=5, color='limegreen', line=dict(color='darkgreen', width=1)),
                             text=df['text'].str[:40]))
fig_b.add_trace(go.Scatter3d(x=C3_b[:, 0], y=C3_b[:, 1], z=C3_b[:, 2], mode='markers',
                             marker=dict(size=10, color='red', line=dict(color='darkred', width=1))))
fig_b.update_layout(height=700, width=900, margin=dict(l=0, r=0, t=40, b=0), showlegend=False,
                    title=f'BIRCH: RI = {rib:.3f}   ARI = {arib:.3f}    Silhouette  = {silb:.3f}')
fig_b.show()

In [ ]:
pca = PCA(n_components=3) # Центрирование. Из каждого столбца вычитается его среднее, чтобы облако точек стояло в начале координат.
#Поиск направлений. Вычисляются главные оси разброса (через SVD, сингулярное разложение матрицы). Каждая ось — это набор из 670 весов, по одному на каждое слово.
#Проекция. Каждый рецепт проецируется на эти 3 оси. Получается 3 числа на рецепт, это и есть строка в X3.
X3 = pca.fit_transform(X)
print(pca.explained_variance_ratio_)        # доля разброса на каждую ось
print(pca.explained_variance_ratio_.sum())  # сколько всего сохранили три оси

rows = []
for i, comp in enumerate(pca.components_): #цикл по матрице (3 на 670), для каждой компоненты вычисляются самые значимые слова
    pos = np.argsort(comp)[::-1][:5]     # 5 слов с самым большим весом
    neg = np.argsort(comp)[:5]           # 5 слов с самым отрицательным весом
    rows.append({
        'компонента': i + 1,
        'доля разброса': f'{pca.explained_variance_ratio_[i]:.1%}',
        '+ конец оси': ', '.join(words[j] for j in pos),
        '− конец оси': ', '.join(words[j] for j in neg),
    })

pd.DataFrame(rows).set_index('компонента')

[0.09298818 0.06725977 0.05822544]
0.2184733857345644


,доля разброса,+ конец оси,− конец оси
компонента,,,
1,9.3%,"grams, sugar, chocolate, tin, cool","pan, fish, cook, garlic, sauce"
2,6.7%,"serving, bread, cheese, layer, cut","chocolate, minutes, pan, pour, cookies"
3,5.8%,"prawns, skewers, sponge, cake, lime","chops, lemon, heat, salmon, filling"
